# Notebook 02 — A/B Test

## Business Question
Did the pricing pilot cause a statistically significant revenue lift in the three treated departments, or could the observed difference be random chance?

## Method
Classic two-sample t-test treating stores as the unit of analysis.
Pilot stores (n=8) = treatment group. Control stores (n=11) = control group.

## Structure
1. **A/A Test** — split control stores into two halves, verify no significant difference in pre-period revenue. Validates the t-test framework before using it.
2. **Power Analysis** — given n=8 pilot stores, MDE=5%, power=80%, what is the smallest effect we could reliably detect?
3. **A/B Test** — one t-test per pilot department during Apr–Jun 2024. Is the lift statistically significant?
4. **Summary** — results table with lift %, p-value, and significance flag per department.

## Key Assumptions
- Stores are independent units (no spillover between stores — SUTVA holds)
- Revenue is approximately normally distributed within each group
- Store 9 (online) excluded throughout

## Limitation
T-test does not control for pre-period differences between stores or time trends.

## Data
Input: `data/analytical_table.csv` (built in data_prep)

# Load Data

In [43]:
# ── Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [44]:
analytical_table = pd.read_csv('../data/analytical_table.csv',parse_dates=['week_start_date'])

In [45]:
analytical_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 9600 entries, 0 to 9599
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   store_id                9600 non-null   int64         
 1   dept_id                 9600 non-null   int64         
 2   week_start_date         9600 non-null   datetime64[us]
 3   during_pilot            9600 non-null   bool          
 4   post_pilot              9600 non-null   bool          
 5   treated                 9600 non-null   bool          
 6   weekly_revenue          9600 non-null   float64       
 7   weekly_units            9600 non-null   int64         
 8   store_name              9600 non-null   str           
 9   city                    9600 non-null   str           
 10  is_pilot                9600 non-null   bool          
 11  store_format            9600 non-null   str           
 12  matched_control_id      3840 non-null   float64       
 13 

In [46]:
analytical_table.head()

,store_id,dept_id,week_start_date,during_pilot,post_pilot,treated,weekly_revenue,weekly_units,store_name,city,is_pilot,store_format,matched_control_id,dept_name,dept_pilot,price_elasticity_group
0,1,1,2023-03-27,False,False,False,25122.0,174,Kitchener-Fairview,Kitchener,True,large,12.0,Electronics,True,Low
1,1,1,2023-04-03,False,False,False,66348.0,463,Kitchener-Fairview,Kitchener,True,large,12.0,Electronics,True,Low
2,1,1,2023-04-10,False,False,False,78808.0,547,Kitchener-Fairview,Kitchener,True,large,12.0,Electronics,True,Low
3,1,1,2023-04-17,False,False,False,76063.0,529,Kitchener-Fairview,Kitchener,True,large,12.0,Electronics,True,Low
4,1,1,2023-04-24,False,False,False,77799.0,542,Kitchener-Fairview,Kitchener,True,large,12.0,Electronics,True,Low


In [47]:
# Exclude Store 9 - online pricing
df = analytical_table[analytical_table['store_id'] != 9].copy()

In [48]:
# Defne basiline period and pilot periods

PILOT_START = pd.Timestamp("2024-04-01")
PILOT_END   = pd.Timestamp("2024-06-30")
PRE_START   = pd.Timestamp("2023-04-01")
PRE_END     = pd.Timestamp("2024-03-31")

print(f"Shape: {df.shape}")
print(f"Pilot stores:   {df[df['is_pilot']==True]['store_id'].nunique()}")
print(f"Control stores: {df[df['is_pilot']==False]['store_id'].nunique()}")

Shape: (9120, 16)
Pilot stores:   8
Control stores: 11


# A/A Test
**Split control stores into two random halves and compare their**

**pre-period revenue. Expect p > 0.05 — no significant difference.**

**If this fails, the control group is broken before we even start.**

In [49]:
pre_period  = df[(df['week_start_date'] >= PRE_START) & (df['week_start_date'] <= PRE_END) & (df['is_pilot'] == False) ]

# Aggregate to weekly revenue per store (all depts combined)
control_weekly = (pre_period.groupby(['store_id', 'week_start_date'])['weekly_revenue'].sum().reset_index())

# Split control stores into two random halves
np.random.seed(42)
control_store_ids = control_weekly['store_id'].unique()
half              = len(control_store_ids) // 2
group_a_stores    = np.random.choice(control_store_ids, size=half, replace=False)
group_b_stores    = np.array([s for s in control_store_ids if s not in group_a_stores])


In [50]:
print(group_a_stores)
print(group_b_stores)

[15 10 19 20 12]
[11 13 14 16 17 18]


In [51]:
# Extract revenue and run A/A t-test
group_a_rev = control_weekly[control_weekly['store_id'].isin(group_a_stores)]['weekly_revenue']
group_b_rev = control_weekly[control_weekly['store_id'].isin(group_b_stores)]['weekly_revenue']

t_stat, p_val = ttest_ind(group_a_rev, group_b_rev)


print("=" * 55)
print("A/A TEST RESULTS")
print("=" * 55)
print(f"Group A stores:  {sorted(group_a_stores)}")
print(f"Group B stores:  {sorted(group_b_stores)}")
print(f"Group A mean:    CAD {group_a_rev.mean():,.0f}")
print(f"Group B mean:    CAD {group_b_rev.mean():,.0f}")
print(f"t-statistic:     {t_stat:.4f}")
print(f"p-value:         {p_val:.4f}")
print(f"Result: {'✅ PASS — control group is clean (p > 0.05)' if p_val > 0.05 else ' FAIL — significant difference detected'}")

A/A TEST RESULTS
Group A stores:  [np.int64(10), np.int64(12), np.int64(15), np.int64(19), np.int64(20)]
Group B stores:  [np.int64(11), np.int64(13), np.int64(14), np.int64(16), np.int64(17), np.int64(18)]
Group A mean:    CAD 175,182
Group B mean:    CAD 194,942
t-statistic:     -3.1723
p-value:         0.0016
Result:  FAIL — significant difference detected


### A/A Test Result — FAIL (p = 0.0016)

The A/A test failed — Group A (CAD 175,182) and Group B (CAD 194,942) show a
statistically significant difference despite both being control stores.

**Root cause:** Small sample size (n=11 control stores). Random 5/6 split by
chance assigned larger stores to Group B and smaller stores to Group A.
With only 11 stores, random halving cannot guarantee size-balanced groups.
This is a sample size limitation.

**Implication for A/B test:** Do not use random store assignment.
Instead use the matched pairs from Notebook 01 — each pilot store is compared
only to its most similar control store, controlling for size and revenue baseline.

**In production:** With hundreds of stores, propensity score matching via
logistic regression would be used. At n=20 stores, Euclidean distance
matching on z-scored features is the appropriate method.

# A/B Test

## Power Analysis

In [52]:
# ── Power Analysis ───────────────────────────────────────────────────
# How small an effect can we reliably detect with n=8 pilot stores?
# If MDE > true effect, insignificant A/B results are expected.



n_pilot, n_control = 8, 11
alpha, power = 0.05, 0.80

# Calculate minimum detectable effect size (Cohen's d)
z_alpha = norm.ppf(1 - alpha/2)  # 1.96
z_beta  = norm.ppf(power)         # 0.84
mde_d   = (z_alpha + z_beta) * np.sqrt(1/n_pilot + 1/n_control)

# Convert Cohen's d to % using pre-period revenue baseline
pre_revenue = df[
    (df['week_start_date'] >= PRE_START) &
    (df['week_start_date'] <= PRE_END)
].groupby(['store_id', 'week_start_date'])['weekly_revenue'].sum()

mde_pct = (mde_d * pre_revenue.std() / pre_revenue.mean()) * 100

print("=" * 50)
print("POWER ANALYSIS")
print("=" * 50)
print(f"Pilot stores (n):      {n_pilot}")
print(f"Control stores (n):    {n_control}")
print(f"Alpha:                 {alpha}")
print(f"Power:                 {power}")
print(f"MDE (Cohen's d):       {mde_d:.3f}")
print(f"MDE as % lift:         {mde_pct:.1f}%")
print(f"Industry threshold:    5%")
print(f"\nResult: {'Adequately powered' if mde_pct <= 5 else f' Underpowered — MDE is {mde_pct:.1f}%, above 5% threshold'}")

POWER ANALYSIS
Pilot stores (n):      8
Control stores (n):    11
Alpha:                 0.05
Power:                 0.8
MDE (Cohen's d):       1.302
MDE as % lift:         56.1%
Industry threshold:    5%

Result:  Underpowered — MDE is 56.1%, above 5% threshold


## A/B Test

In [53]:
# ── A/B Test using matched pairs ─────────────────────────────────────
# Use matched_control_id from stores to compare each pilot store
# only against its closest control store — eliminates size imbalance

pilot_period = df[
    (df['week_start_date'] >= PILOT_START) &
    (df['week_start_date'] <= PILOT_END)
]

# Get matched control store IDs for pilot stores
matched_controls = df[df['is_pilot'] == True][['store_id', 'matched_control_id']].drop_duplicates()
matched_control_ids = matched_controls['matched_control_id'].dropna().astype(int).tolist()

PILOT_DEPTS = {1: 'Electronics', 2: 'Home & Kitchen', 3: 'Sports & Outdoors'}

results = []
print("=" * 60)
print("A/B TEST — Matched Pairs (Pilot vs Matched Control)")
print("Pilot period: Apr–Jun 2024")
print("=" * 60)

for dept_id, dept_name in PILOT_DEPTS.items():
    dept_data = pilot_period[pilot_period['dept_id'] == dept_id]

    pilot_rev   = dept_data[dept_data['is_pilot'] == True]['weekly_revenue']
    control_rev = dept_data[dept_data['store_id'].isin(matched_control_ids)]['weekly_revenue']
    
    t_stat, p_val = ttest_ind(pilot_rev, control_rev)
    lift_pct = ((pilot_rev.mean() - control_rev.mean()) / control_rev.mean()) * 100

    results.append({
        'Department':   dept_name,
        'Pilot Mean':   round(pilot_rev.mean(), 0),
        'Control Mean': round(control_rev.mean(), 0),
        'Lift %':       round(lift_pct, 2),
        'p-value':      round(p_val, 4),
        'Significant':  'Yes' if p_val < 0.05 else 'No'
    })

    print(f"\n{dept_name}")
    print(f"  Pilot mean:   CAD {pilot_rev.mean():,.0f}")
    print(f"  Control mean: CAD {control_rev.mean():,.0f}")
    print(f"  Lift:         {lift_pct:+.2f}%")
    print(f"  p-value:      {p_val:.4f}")
    print(f"  Significant:  {'Yes' if p_val < 0.05 else 'No'}")

results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print(results_df.to_string(index=False))


A/B TEST — Matched Pairs (Pilot vs Matched Control)
Pilot period: Apr–Jun 2024

Electronics
  Pilot mean:   CAD 61,755
  Control mean: CAD 56,490
  Lift:         +9.32%
  p-value:      0.1171
  Significant:  No

Home & Kitchen
  Pilot mean:   CAD 38,638
  Control mean: CAD 38,341
  Lift:         +0.77%
  p-value:      0.8910
  Significant:  No

Sports & Outdoors
  Pilot mean:   CAD 24,558
  Control mean: CAD 25,539
  Lift:         -3.84%
  p-value:      0.4852
  Significant:  No

       Department  Pilot Mean  Control Mean  Lift %  p-value Significant
      Electronics     61755.0       56490.0    9.32   0.1171          No
   Home & Kitchen     38638.0       38341.0    0.77   0.8910          No
Sports & Outdoors     24558.0       25539.0   -3.84   0.4852          No


### A/B Test Results — Key Insights

**All three departments show insignificant results (p > 0.05).** However this is
expected given the power analysis finding — the test is underpowered.

- **Electronics** showed the strongest directional signal: +9.32% lift (p=0.1171).
  Direction is correct — ground truth ATT is positive. Insignificance is due to
  small sample size.

- **Home & Kitchen** showed minimal lift: +0.77% (p=0.8910). Consistent with
  the small true ATT expected for medium-elasticity departments.

- **Sports & Outdoors** showed a negative lift: -3.84% (p=0.4852). Direction
  is correct — high elasticity department expected to lose revenue from price change.

**Why results are insignificant despite correct direction:**
With MDE = 56.1%, the t-test needs to see a 56% lift to call it significant.
The true effects (12%, 3%, -2%) are far below this threshold. This is a
fundamental limitation of small-sample A/B testing.

**Conclusion:** A/B test alone is insufficient at n=8 stores. The directionally
correct results motivate using more powerful causal methods — DiD ,
Synthetic Control , and BSTS — which control for
pre-period trends and store-level differences to extract signal from small samples.

**How to properly conduct an A/B test with only 8 pilot and 11 control stores:**

1. **Run the test longer** — extend pilot to 6–12 months. More observations
   per store reduce variance and lower the MDE.
2. **Use CUPED** — incorporate pre-period revenue as a covariate to reduce
   variance by 30–50% without adding stores. Industry standard at Netflix,
   Booking, and Microsoft.
3. **Switch to paired t-test** — compare each pilot store directly to its
   matched control store as a pair. More powerful than independent samples t-test.
4. **Use one-tailed test** — if business has a directional hypothesis,
   one-tailed test reduces MDE by approximately 20%.

**Other methods**
   - **Mann-Whitney U** — non-parametric alternative, does not assume normality,
  suitable when revenue distribution is skewed
- **Permutation test** — assumption-free, most appropriate for n=8 small samples,
  makes no distributional assumptions at all
- **Paired t-test** — compares each pilot store directly to its matched control
  store as a pair, more powerful than independent samples t-test at small n

T-test (independent samples) was chosen for interpretability and consistency
with industry standard A/B test reporting at store level.